In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 单候选中间潜空间写入—读取闭环

A是有限探索，B内容自适应是主线。当前候选未证明解决V4/V6的传递或识别失败；本地测试不能当作潜空间同步成功。

默认仅打印计划。一次真实检查为3次生成、4张最终图（plain、content-only、固定anchor、v2 RGB同步参考），三方法×正负×clean/+10°=12条完整盲路径；另对无anchor内容参考做2次reader诊断。旋转使用现v2 renderer。没有oracle扫描、候选选择、尺度/全局RST、Attention或浅反演。

执行需要本轮外部授权。将交付的latent_sync_v1.zip放到下述路径，或将源码解包到REPO；本次分支尚未推送。

In [ ]:
from pathlib import Path
import os, sys, subprocess, shutil, json, datetime
REPO = Path('/content/latent-sync-v1')
SOURCE_ARCHIVE = Path('/content/drive/MyDrive/CEG-WM/development/latent_sync_v1.zip')
if not REPO.exists():
    shutil.unpack_archive(str(SOURCE_ARCHIVE), '/content')
EXECUTE = False
UNIT_INDEX = 0
unit = json.loads((REPO / 'configs/parallel_method_dev/fit.json').read_text())[UNIT_INDEX]
OUTPUT = Path('/content') / ('latent-readability-' + datetime.datetime.now().strftime('%Y%m%d-%H%M%S'))
print({'execute': EXECUTE, 'unit': unit['id'], 'output': str(OUTPUT)})


## 环境与凭据
版本只记录，不限定GPU型号。默认不读取凭据、不加载模型。输出写入Colab本地。

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(REPO), 'scipy', 'accelerate', 'lpips', 'torchmetrics'], check=True)
if EXECUTE:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    os.environ['CEG_WM_ROOT_KEY'] = userdata.get('CEG_WM_ROOT_KEY')
import importlib.metadata as metadata
print({name: metadata.version(name) for name in ['torch', 'diffusers', 'transformers', 'scipy']})


In [ ]:
command = [sys.executable, '-m', 'experiments.run_latent_sync_development',
           '--output', str(OUTPUT), '--seed', str(unit['seed']), '--prompt', unit['prompt']]
if EXECUTE:
    command += ['--execute']
subprocess.run(command, cwd=REPO, check=True)


## 读取结果与停止边界

检查report.json、rows.jsonl和四张图：同一个公共reader在anchor/no-anchor上的相关、角度与歧义；实际post及完整路径内容正负分离；PSNR/SSIM/LPIPS与局部失真。v2 RGB同步仅作保留的可运行参考，A自身读出不使用它。

生产读出只用当前RGB、公共VAE/模板与密钥内容评分，不读同图clean、原latent或embed residual。配对差值仅在开发报告中计算；自身oracle-post差小不能选优，旧多候选selector已改为描述性比较。

若真实clean仍不可读，或仅私有参考可读，则停止本候选并将资源留给B；不增加搜索/模板或放宽阈值。当前尚无能区别于历史失败的已验证读出修复。既有容差、renderer、subpixel结果先复用，crop/重建保留长期研究范围。

本notebook仅完成结构与Python语法验证。Colab挂载、依赖安装、模型与真实结果尚未执行；外部授权后逐格验证。